# YOLO K-Fold 교차검증 — 그룹화 fix 버전 (run_experiment_kfold)

`exp020`(콤보 문자열 기준 K-Fold)의 근접중복 감사에서 **fold마다 val의 28~35%
(평균 32%)** 가 "조합 문자열은 다르지만 같은 트레이/배경에서 약 1개만 바꿔 찍은"
근접중복으로 확인됐다. 5개 fold 모두 비슷한 비율로 나왔고, mAP@75:95도
0.9602~0.9719(평균 0.9669, 표준편차 0.0044)로 exp011 단일 split(0.9914)보다
일관되게 낮았다 — 두 문제(작은 val 표본 + 근접중복)가 exp011의 점수를
부풀렸다는 정황이다.

이 노트북은 `build_leakage_safe_groups()`로 근접중복 장면까지 하나의 그룹으로
묶은 뒤 K-Fold를 다시 돈다(**exp021**). exp020은 "그룹화 전" 비교 기준으로
그대로 남겨둔다.

추가로, AIHub 조합 이미지 중 **실제 Kaggle 테스트셋**과 근접중복인 것(6장 발견,
`audit_leakage.audit()`)도 K-Fold 풀을 만들기 전에 미리 제외한다 — 이건 fold
내부 분할과 무관하게 존재하는, 더 심각한(진짜 컨닝에 가까운) 문제였다.

확인 절차:
1. Kaggle train(56) + AIHub 조합[1,3]에서 실제 테스트셋 근접중복 이미지를 먼저 제외.
2. 남은 풀 전체에 대해 `build_leakage_safe_groups`로 누수 안전 그룹을 만든다.
3. 이 그룹 기준으로 K-Fold 분할(`create_group_kfold(..., group_overrides=...)`).
4. fold마다 YOLO를 학습·평가하고, exp020(그룹화 전) 결과와 비교한다.
5. fold 내부 근접중복 재검사로 그룹화가 실제로 효과가 있었는지 확인한다(0에
   가까워야 함).

새 파일로 유지한 이유: 실행 시간이 fold 수만큼 배수라 별도로 관리하는 게 편하다.

In [ ]:
from utils.sys_utils import add_experiment_root

PROJECT_ROOT = add_experiment_root()
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.audit_leakage import _dhash, _gray_vec, _rmse, audit
from src.data.aihub_converter import convert_aihub_to_coco
from src.data.coco_to_yolo import prepare_yolo_dataset
from src.data.kaggle_converter import convert_kaggle_to_coco
from src.data.merge import merge_for_augmentation
from src.data.subset import (
    build_leakage_safe_groups,
    combo_group_key,
    create_group_kfold,
    exclude_images,
)
from src.train_yolo import run_yolo_experiment
from src.utils import load_config

CONFIGS_DIR = PROJECT_ROOT / "configs" / "experiment"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "kfold"
DATA_DIR = PROJECT_ROOT / "data" / "kfold_clean"
AIHUB_COMBOS = [1, 3]
N_SPLITS = 5

print("EXPERIMENTS_DIR:", EXPERIMENTS_DIR)
print("DATA_DIR:", DATA_DIR)


## 1단계 — 데이터 풀 구성 + 실제 테스트셋 leakage 필터링

`audit_leakage.audit()`로 실제 Kaggle 테스트셋과 근접중복인 AIHub 이미지를 먼저
찾아 제외한다(fold 분할과 무관한, 더 근본적인 문제).

In [ ]:
kaggle_coco = convert_kaggle_to_coco()
allowed_ids = {c["id"] for c in kaggle_coco["categories"]}
aihub_coco_raw = convert_aihub_to_coco(combo_nums=AIHUB_COMBOS)

audit_result = audit(combos=AIHUB_COMBOS)
print("\n판정:", audit_result["verdict"])
print(
    "AIHub 근접중복(실제 테스트셋과):",
    len(audit_result["aihub_leak_paths"]),
    "장 → 제외 예정",
)

aihub_coco = exclude_images(aihub_coco_raw, audit_result["aihub_leak_paths"])


## 2단계 — 근접중복 인식 그룹 구성 + 전체 풀 병합

`merge_for_augmentation`에 빈 val을 넘겨 "제외 로직 없이 Kaggle 전체 + AIHub를
하나로 합치기"만 재사용한다(실제 train/val 분할은 K-Fold에서 수행). 그룹은
`build_leakage_safe_groups`로 미리 계산해 병합 시 누수 검사에도, 이후 K-Fold
분할에도 동일하게 사용한다.

In [ ]:
_pool_images = kaggle_coco["images"] + aihub_coco["images"]
group_overrides = build_leakage_safe_groups({
    "images": _pool_images,
    "annotations": [],
    "categories": [],
})

_empty_val = {"images": [], "annotations": [], "categories": kaggle_coco["categories"]}
combined_coco, _ = merge_for_augmentation(
    base_train=kaggle_coco,
    base_val=_empty_val,
    extra_coco=aihub_coco,
    allowed_ids=allowed_ids,
    drop_pure_irrelevant=True,
    group_overrides=group_overrides,
)
print(
    f"전체 풀: 이미지 {len(combined_coco['images'])}장, 클래스 {len(combined_coco['categories'])}개"
)


## 3단계 — 그룹(콤보+근접중복) 단위 K-Fold 분할

`group_overrides`를 넘겨 근접중복 장면이 fold 경계를 넘지 않도록 한다. 내부에서
(a) fold별 train/val 그룹 누수 0, (b) fold 간 val 그룹 중복 0, (c) 전체 그룹
커버리지를 assert로 강제한다.

In [ ]:
folds = create_group_kfold(
    combined_coco, n_splits=N_SPLITS, seed=42, group_overrides=group_overrides
)
print(f"{len(folds)}개 fold 생성 완료")


### 추가 검증 — 파티션 무결성 재확인 (그룹화 fix 반영)

라이브러리 함수의 assert를 신뢰하되, 노트북에서도 같은 논리를 독립적으로 다시
계산한다. 이번엔 `combo_group_key` 대신 `group_overrides`(근접중복 병합 반영)로
그룹을 계산해야 앞 단계와 일관된 검증이 된다.

In [ ]:
def group_of(file_name):
    return group_overrides.get(file_name, combo_group_key(file_name))


all_groups = {group_of(img["file_name"]) for img in combined_coco["images"]}
val_group_sets = []
for i, (tr, va) in enumerate(folds):
    va_groups = {group_of(img["file_name"]) for img in va["images"]}
    tr_groups = {group_of(img["file_name"]) for img in tr["images"]}
    assert not (va_groups & tr_groups), f"fold {i}: train/val 그룹 겹침"
    val_group_sets.append(va_groups)

for i in range(len(val_group_sets)):
    for j in range(i + 1, len(val_group_sets)):
        overlap = val_group_sets[i] & val_group_sets[j]
        assert not overlap, (
            f"fold {i}와 fold {j}의 val 그룹이 겹침: {list(overlap)[:3]}"
        )

union_val_groups = set().union(*val_group_sets)
missing = all_groups - union_val_groups
assert not missing, f"K-Fold가 커버하지 못한 그룹 {len(missing)}개"
print(
    f"파티션 무결성 확인 완료: 전체 {len(all_groups)}개 그룹이 "
    "fold마다 겹침 없이 정확히 한 번씩 val로 사용됨"
)


### 검증 — 지각 해시 근접중복 재점검 (그룹화 fix 전/후 비교)

exp020(그룹화 전)에서는 fold마다 val의 28~35%가 근접중복으로 잡혔다. 그룹화가
제대로 됐다면 이번엔 0에 가까워야 한다.

In [ ]:
def audit_fold_near_duplicates(
    train_coco, val_coco, hash_size=16, hash_cutoff=12, rmse_cutoff=0.06
):
    """fold의 train/val 사이 지각 해시 근접 중복(같은 사진일 가능성)을 스캔."""
    train_paths = [Path(im["file_name"]) for im in train_coco["images"]]
    val_paths = [Path(im["file_name"]) for im in val_coco["images"]]

    train_dh = []
    for p in train_paths:
        d = _dhash(p, size=hash_size)
        if d is not None:
            train_dh.append((d, p))

    near_dups = []
    affected_val = set()
    for vp in val_paths:
        vd = _dhash(vp, size=hash_size)
        if vd is None or not train_dh:
            continue
        best_d, best_p = min(
            ((bin(vd ^ d).count("1"), p) for d, p in train_dh), key=lambda x: x[0]
        )
        if best_d <= hash_cutoff:
            vv = _gray_vec(vp)
            sv = _gray_vec(best_p)
            if vv is not None and sv is not None:
                r = _rmse(vv, sv)
                if r <= rmse_cutoff:
                    near_dups.append((vp, best_p, best_d, r))
                    affected_val.add(vp)

    summary = {
        "n_val": len(val_paths),
        "n_pairs": len(near_dups),
        "n_val_affected": len(affected_val),
        "pct_val_affected": 100 * len(affected_val) / max(len(val_paths), 1),
    }
    return near_dups, summary


fold_audit_summaries = []
for i, (tr, va) in enumerate(folds):
    dups, summary = audit_fold_near_duplicates(tr, va)
    fold_audit_summaries.append({"fold": i, **summary})
    print(
        f"[fold {i}] val {summary['n_val']}장 중 근접중복 의심 "
        f"{summary['n_val_affected']}장({summary['pct_val_affected']:.1f}%) | "
        f"총 매칭 {summary['n_pairs']}쌍"
    )
    for vp, tp, hd, r in dups[:3]:
        print(f"    VAL {vp.name}  ~~  TRAIN {tp.name}  (hash {hd}, rmse {r:.3f})")

audit_summary_df = pd.DataFrame(fold_audit_summaries)
display(audit_summary_df)
print("\n(참고: exp020 그룹화 전에는 fold 평균 32.0%였음)")


## 4단계 — fold별 YOLO 학습 (exp021, exp020의 그룹화 fix 버전)

fold마다 실험 이름에 `_f{i}`를 붙여 저장한다. VSCode/커널이 중간에 끊겨도 이미
끝난 fold는 재학습하지 않고 저장된 `metrics.json`을 재사용한다(멱등적 재실행).

In [ ]:
kfold_cfg_base = load_config(CONFIGS_DIR / "exp021_yolo11n_kfold_clean.yaml")

fold_metrics = []
for i, (train_fold, val_fold) in enumerate(folds):
    cfg = json.loads(json.dumps(kfold_cfg_base))  # 딕셔너리 깊은 복사
    exp_name = f"{kfold_cfg_base['experiment']['name']}_f{i}"
    cfg["experiment"]["name"] = exp_name

    existing_metrics = EXPERIMENTS_DIR / exp_name / "metrics.json"
    if existing_metrics.exists():
        metrics = json.load(open(existing_metrics))
        metrics["fold"] = i
        fold_metrics.append(metrics)
        print(f"[fold {i}] 이미 완료됨({existing_metrics}) — 재학습 건너뜀")
        continue

    yolo_dir = DATA_DIR / f"yolo_fold{i}"
    yaml_path = prepare_yolo_dataset(train_fold, val_fold, yolo_dir, symlink=True)

    print(f"\n=== fold {i} 학습 시작 ({exp_name}) ===")
    metrics = run_yolo_experiment(cfg, yaml_path, EXPERIMENTS_DIR)
    metrics["fold"] = i
    fold_metrics.append(metrics)
    print(
        f"fold {i} 완료: mAP@75={metrics['map75']:.4f}, mAP@75:95={metrics['map75_95']:.4f}"
    )


## 5단계 — 결과 집계 (그룹화 전/후 비교)

exp020(그룹화 전) 대비 exp021(그룹화 후) 평균이 얼마나 달라졌는지가 근접중복이
실제로 점수에 미친 영향의 추정치다. exp011/exp016(단일 split)과도 비교한다.

In [ ]:
kfold_df = pd.DataFrame(fold_metrics)[["fold", "map75", "map75_95"]]
display(kfold_df)

summary = kfold_df[["map75", "map75_95"]].agg(["mean", "std", "min", "max"])
print(summary)

print(
    f"\nexp021(그룹화 후) K-Fold 평균: {kfold_df['map75_95'].mean():.4f} "
    f"(표준편차 {kfold_df['map75_95'].std():.4f})"
)
print("exp020(그룹화 전) K-Fold 평균: 0.9669 (표준편차 0.0044) — 5 fold 완료 기준")

for exp_name, label in [
    ("exp011_yolo11n_aug", "exp011(단일 split, 그룹화 전)"),
    ("exp016_yolo11n_aug_clean", "exp016(단일 split, 그룹화 후)"),
]:
    f = PROJECT_ROOT / "experiments" / exp_name / "metrics.json"
    if f.exists():
        m = json.load(open(f))
        print(f"{label}: mAP@75:95 = {m.get('map75_95', m.get('final_map75_95')):.4f}")
    else:
        print(f"{label}: 아직 학습 안 됨")
